# 01 — Contract analyzer architecture

| | |
|---|---|
| **Track** | Applied Labs |
| **Domain** | AG — Legal Tech |
| **Level** | Advanced |
| **Time** | ~35 min |
| **Prerequisites** | `00_first_principles.ipynb` |
| **Open in Colab** | `labs/03_contract_analyzer/01_architecture.ipynb` |

> **Goal:** Design every component of the contract analyzer — from clause segmentation
> to report generation — with working prototypes for each stage.

## Setup

In [ ]:
# ── Install dependencies ──
!pip install -q "openai>=1.0.0" "sentence-transformers>=2.2.0" "numpy>=1.24.0" "chromadb>=0.4.0"

import os, json, warnings
warnings.filterwarnings("ignore")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if OPENAI_API_KEY:
    print("✓ OPENAI_API_KEY found")
else:
    print("⚠  OPENAI_API_KEY not set — LLM cells will use mock responses")

print("✓ Dependencies installed")

## 1 · System architecture

```
┌─────────────────────────────────────────────────────────────────────┐
│                    CONTRACT ANALYZER PIPELINE                       │
│                                                                     │
│   ┌──────────┐   ┌──────────┐   ┌──────────┐   ┌──────────────┐   │
│   │ Contract │──▶│  Clause  │──▶│  Clause  │──▶│    Risk      │   │
│   │   Text   │   │Segmenter │   │Classifier│   │   Scorer     │   │
│   └──────────┘   └──────────┘   └──────────┘   └──────┬───────┘   │
│                                                        │            │
│                                                        ▼            │
│   ┌──────────┐   ┌──────────────┐   ┌──────────────────────────┐   │
│   │  Report  │◀──│ Negotiation  │◀──│  Standard Comparator     │   │
│   │Generator │   │   Advisor    │   │  (RAG — clause library)  │   │
│   └──────────┘   └──────────────┘   └──────────────────────────┘   │
│                                                                     │
│   Data stores:  [Clause Taxonomy]  [Standard Library]  [Risk DB]   │
└─────────────────────────────────────────────────────────────────────┘
```

Each component is **independently testable** and communicates through
well-defined data structures.

In [ ]:
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple
from enum import Enum

# ── Core data types shared across all components ──

class ClauseType(str, Enum):
    INDEMNIFICATION = "indemnification"
    LIMITATION_OF_LIABILITY = "limitation_of_liability"
    IP_ASSIGNMENT = "ip_assignment"
    TERMINATION = "termination"
    CONFIDENTIALITY = "confidentiality"
    FORCE_MAJEURE = "force_majeure"
    NON_COMPETE = "non_compete"
    PAYMENT_TERMS = "payment_terms"
    WARRANTY = "warranty"
    DISPUTE_RESOLUTION = "dispute_resolution"
    GENERAL = "general"

@dataclass
class Clause:
    """A single clause extracted from a contract."""
    text: str
    index: int
    section_header: Optional[str] = None

@dataclass
class ClassifiedClause:
    """A clause with its predicted type and confidence."""
    clause: Clause
    clause_type: ClauseType
    confidence: float

@dataclass
class ScoredClause:
    """A classified clause with a risk score and explanation."""
    classified: ClassifiedClause
    risk_score: float          # 1.0 (safe) to 5.0 (critical)
    risk_factors: List[str]
    explanation: str

@dataclass
class Deviation:
    """Difference between a clause and its market-standard version."""
    scored_clause: ScoredClause
    standard_text: str
    similarity_score: float
    deviation_summary: str

@dataclass
class NegotiationPoint:
    """Actionable recommendation for a risky clause."""
    deviation: Deviation
    plain_english_risk: str
    suggested_alternative: str
    talking_points: List[str]

@dataclass
class ContractReport:
    """Final output of the pipeline."""
    executive_summary: str
    clause_table: List[Dict]
    top_risks: List[ScoredClause]
    negotiation_points: List[NegotiationPoint]
    metadata: Dict[str, str]

print("Core data types defined:")
for cls in [Clause, ClassifiedClause, ScoredClause, Deviation, NegotiationPoint, ContractReport]:
    print(f"  • {cls.__name__:20s} — {cls.__doc__.strip()}")

## 2 · Clause segmentation

The segmenter splits raw contract text into individual clauses using
**structural cues**: section numbers, paragraph breaks, and legal headings.

Design principles:
- Preserve section headers for downstream classification
- Handle numbered (1.1, 1.2) and lettered (a, b, c) sections
- Keep multi-sentence clauses together

In [ ]:
import re
from typing import List

def segment_contract(text: str) -> List[Clause]:
    """Split contract text into clauses using structural cues.

    Heuristics:
    1. Split on section-number patterns (e.g., '1.', '1.1', 'Section 3')
    2. Split on double newlines (paragraph breaks)
    3. Merge short fragments back into preceding clause
    """
    # Normalize whitespace
    text = re.sub(r'\r\n', '\n', text)

    # Pattern: numbered sections like "1.", "1.1", "Section 3", "ARTICLE IV"
    section_pattern = re.compile(
        r'\n\s*(?='
        r'(?:\d+\.\d*\s)'           # 1. or 1.1
        r'|(?:Section\s+\d+)'         # Section 3
        r'|(?:ARTICLE\s+[IVXLCDM]+)'  # ARTICLE IV
        r'|(?:[A-Z][A-Z\s]{5,}:)'     # ALL-CAPS HEADING:
        r')',
        re.IGNORECASE
    )

    # Split on patterns
    raw_segments = section_pattern.split(text)

    # Also split on double newlines within each segment
    clauses: List[Clause] = []
    idx = 0
    for segment in raw_segments:
        paragraphs = re.split(r'\n\s*\n', segment.strip())
        for para in paragraphs:
            para = para.strip()
            if len(para) < 20:
                # Merge short fragments into previous clause
                if clauses:
                    clauses[-1] = Clause(
                        text=clauses[-1].text + " " + para,
                        index=clauses[-1].index,
                        section_header=clauses[-1].section_header,
                    )
                continue

            # Extract section header if present
            header_match = re.match(
                r'^(\d+\.\d*\s+[A-Za-z ]+|Section\s+\d+[^.]*|ARTICLE\s+[IVXLCDM]+[^.]*)',
                para
            )
            header = header_match.group(0).strip() if header_match else None

            clauses.append(Clause(text=para, index=idx, section_header=header))
            idx += 1

    return clauses

# ── Test on a sample contract ──
SAMPLE_CONTRACT = """
1. DEFINITIONS
"Agreement" means this Software License Agreement between TechVendor Inc.
("Vendor") and GlobalCorp Ltd. ("Client"), effective as of January 1, 2025.

1.1 "Software" means the proprietary platform described in Exhibit A,
including all updates and patches delivered during the term.

2. LICENSE GRANT
Vendor grants Client a non-exclusive, non-transferable license to use the
Software for Client's internal business purposes only. Client shall not
sublicense, modify, or reverse-engineer the Software.

3. PAYMENT TERMS
Client shall pay the annual license fee of $500,000 within 30 days of each
anniversary date. Late payments accrue interest at 1.5% per month.

4. CONFIDENTIALITY
Each party shall maintain the confidentiality of the other party's proprietary
information for a period of five years following disclosure. Standard exceptions
apply: information that becomes publicly available, was independently developed,
or is required to be disclosed by law.

5. INDEMNIFICATION
Vendor shall indemnify, defend, and hold harmless Client from any third-party
claims arising from Vendor's infringement of intellectual property rights, provided
Client gives prompt notice and reasonable cooperation.

6. LIMITATION OF LIABILITY
IN NO EVENT SHALL EITHER PARTY'S AGGREGATE LIABILITY EXCEED THE TOTAL FEES
PAID OR PAYABLE UNDER THIS AGREEMENT IN THE TWELVE MONTHS PRECEDING THE CLAIM.
NEITHER PARTY SHALL BE LIABLE FOR INDIRECT, CONSEQUENTIAL, OR PUNITIVE DAMAGES.

7. TERMINATION
Either party may terminate this Agreement with 90 days written notice. Upon
termination, Client shall cease all use of the Software and return or destroy
all copies. Sections 4, 5, and 6 shall survive termination.
"""

clauses = segment_contract(SAMPLE_CONTRACT)
print(f"Segmented {len(clauses)} clauses:\n")
for c in clauses:
    header = f" [{c.section_header}]" if c.section_header else ""
    print(f"  Clause {c.index}{header}")
    print(f"    {c.text[:100]}…" if len(c.text) > 100 else f"    {c.text}")
    print()

## 3 · Clause classification

We classify each clause by comparing its embedding against **labeled
exemplars** for each clause type.  This is a **few-shot embedding classifier**:
no fine-tuning needed, just representative examples.

For each clause type, we store 2–3 exemplar sentences.  At inference time
we find the closest exemplar and assign that type.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

# ── Exemplars for each clause type ──
EXEMPLARS: Dict[str, List[str]] = {
    "indemnification": [
        "The vendor shall indemnify the client against third-party claims.",
        "Each party agrees to hold harmless and defend the other from losses.",
    ],
    "limitation_of_liability": [
        "Neither party's liability shall exceed fees paid in the prior 12 months.",
        "In no event shall either party be liable for consequential damages.",
    ],
    "ip_assignment": [
        "All work product shall be owned by the client as work made for hire.",
        "Vendor assigns all intellectual property rights in deliverables to client.",
    ],
    "termination": [
        "Either party may terminate with 60 days written notice.",
        "This agreement automatically terminates upon material breach.",
    ],
    "confidentiality": [
        "Each party shall keep confidential information secret for five years.",
        "Receiving party shall not disclose proprietary information to third parties.",
    ],
    "force_majeure": [
        "Neither party is liable for failure due to acts of God or natural disaster.",
        "Force majeure events excuse performance for the duration of the event.",
    ],
    "payment_terms": [
        "Client shall pay invoices within 30 days of receipt.",
        "Late payments accrue interest at 1.5% per month.",
    ],
    "warranty": [
        "Vendor warrants that the software will perform materially as described.",
        "The services are provided as-is without warranty of any kind.",
    ],
    "dispute_resolution": [
        "Disputes shall be resolved by binding arbitration in New York.",
        "The parties agree to attempt mediation before pursuing litigation.",
    ],
    "general": [
        "This agreement shall be governed by the laws of Delaware.",
        "Notices shall be delivered to the addresses in Exhibit B.",
    ],
}

# Pre-compute exemplar embeddings
exemplar_data: List[Tuple[str, np.ndarray]] = []
for ctype, examples in EXEMPLARS.items():
    embs = model.encode(examples, show_progress_bar=False)
    for emb in embs:
        exemplar_data.append((ctype, emb))

exemplar_types = [e[0] for e in exemplar_data]
exemplar_matrix = np.array([e[1] for e in exemplar_data])

def classify_clause(clause_text: str) -> Tuple[str, float]:
    """Classify clause by nearest-exemplar in embedding space."""
    emb = model.encode([clause_text], show_progress_bar=False)[0]
    # Cosine similarity
    sims = np.dot(exemplar_matrix, emb) / (
        np.linalg.norm(exemplar_matrix, axis=1) * np.linalg.norm(emb)
    )
    best_idx = int(np.argmax(sims))
    return exemplar_types[best_idx], float(sims[best_idx])

# ── Test on 15 clauses ──
TEST_CLAUSES_CLASSIFY: List[Tuple[str, str]] = [
    ("Vendor shall indemnify Client against all IP infringement claims.", "indemnification"),
    ("Total aggregate liability is capped at twelve months of fees.", "limitation_of_liability"),
    ("All inventions created during the engagement belong to Client.", "ip_assignment"),
    ("Either party may terminate with 30 days written notice.", "termination"),
    ("Confidential information must not be shared with competitors.", "confidentiality"),
    ("No party is liable for delays caused by pandemic or natural disaster.", "force_majeure"),
    ("Invoices are due net-45 from the date of receipt.", "payment_terms"),
    ("The software is warranted to be free of material defects for one year.", "warranty"),
    ("Any disputes will be settled by arbitration under ICC rules.", "dispute_resolution"),
    ("This agreement is governed by California law.", "general"),
    ("Provider will compensate recipient for any losses from data breach.", "indemnification"),
    ("Damages shall never exceed the contract value.", "limitation_of_liability"),
    ("The receiving party shall return all proprietary materials on termination.", "confidentiality"),
    ("Service fees are $10,000 monthly, payable on the first of each month.", "payment_terms"),
    ("This contract ends automatically if either party enters bankruptcy.", "termination"),
]

correct = 0
print(f"{'Clause (truncated)':<60} {'Expected':<26} {'Predicted':<26} {'Conf':>5} {'OK'}")
print("─" * 125)
for text, expected in TEST_CLAUSES_CLASSIFY:
    predicted, conf = classify_clause(text)
    match = "✓" if predicted == expected else "✗"
    if predicted == expected:
        correct += 1
    short = text[:57] + "…" if len(text) > 57 else text
    print(f"{short:<60} {expected:<26} {predicted:<26} {conf:>5.2f} {match}")

print(f"\nAccuracy: {correct}/{len(TEST_CLAUSES_CLASSIFY)} = {correct/len(TEST_CLAUSES_CLASSIFY):.0%}")

## 4 · Risk scoring

Risk scoring uses a **multi-factor weighted formula**:

| Factor | Weight | What it measures |
|---|---|---|
| Clause-type baseline | 0.20 | Some types are inherently riskier (indemnification > definitions) |
| Party asymmetry | 0.30 | One party bears disproportionate burden |
| Language vagueness | 0.25 | Undefined terms, ambiguous scope |
| Deviation from standard | 0.25 | How far from market-norm wording |

Final score: weighted sum → clamped to [1.0, 5.0].

In [ ]:
from typing import NamedTuple

class RiskFactors(NamedTuple):
    type_baseline: float     # 1-5
    party_asymmetry: float   # 1-5
    language_vagueness: float # 1-5
    deviation_score: float   # 1-5

WEIGHTS = {
    "type_baseline": 0.20,
    "party_asymmetry": 0.30,
    "language_vagueness": 0.25,
    "deviation_score": 0.25,
}

TYPE_BASELINES: Dict[str, float] = {
    "indemnification": 3.5,
    "limitation_of_liability": 3.0,
    "ip_assignment": 3.0,
    "termination": 2.5,
    "confidentiality": 2.0,
    "force_majeure": 1.5,
    "payment_terms": 2.0,
    "warranty": 2.0,
    "dispute_resolution": 1.5,
    "general": 1.0,
}

def compute_risk_score(factors: RiskFactors) -> float:
    """Compute weighted risk score clamped to [1.0, 5.0]."""
    raw = (
        WEIGHTS["type_baseline"]      * factors.type_baseline +
        WEIGHTS["party_asymmetry"]    * factors.party_asymmetry +
        WEIGHTS["language_vagueness"] * factors.language_vagueness +
        WEIGHTS["deviation_score"]    * factors.deviation_score
    )
    return max(1.0, min(5.0, raw))

# ── Score 10 example clauses ──
EXAMPLE_CLAUSES_RISK: List[Tuple[str, str, RiskFactors]] = [
    ("Vendor indemnifies Client without any cap on liability",
     "indemnification", RiskFactors(3.5, 4.5, 3.0, 4.0)),
    ("Each party's liability capped at 12 months of fees",
     "limitation_of_liability", RiskFactors(3.0, 1.5, 1.0, 1.0)),
    ("All IP, including pre-existing, assigned to Client",
     "ip_assignment", RiskFactors(3.0, 5.0, 2.0, 4.5)),
    ("Either party terminates with 60 days notice",
     "termination", RiskFactors(2.5, 1.0, 1.0, 1.0)),
    ("Confidentiality lasts in perpetuity, no exceptions, non-mutual",
     "confidentiality", RiskFactors(2.0, 4.0, 3.5, 4.0)),
    ("Force majeure covers pandemic and natural disaster only",
     "force_majeure", RiskFactors(1.5, 1.0, 1.0, 1.0)),
    ("Payment net-30, 1.5% monthly interest on late payments",
     "payment_terms", RiskFactors(2.0, 2.0, 1.0, 1.5)),
    ("Software warranted for 12 months; sole remedy is re-performance",
     "warranty", RiskFactors(2.0, 1.5, 1.5, 1.0)),
    ("Client may terminate at will; Vendor cannot terminate under any circumstances",
     "termination", RiskFactors(2.5, 5.0, 2.0, 5.0)),
    ("Disputes resolved by binding arbitration in Client's home jurisdiction",
     "dispute_resolution", RiskFactors(1.5, 3.0, 1.5, 2.5)),
]

print(f"{'Clause (truncated)':<60} {'Type':<26} {'Score':>6} {'Level'}")
print("─" * 100)
for text, ctype, factors in EXAMPLE_CLAUSES_RISK:
    score = compute_risk_score(factors)
    level = "🟢 Low" if score < 2.0 else "🟡 Medium" if score < 3.0 else "🟠 High" if score < 4.0 else "🔴 Critical"
    short = text[:57] + "…" if len(text) > 57 else text
    print(f"{short:<60} {ctype:<26} {score:>5.2f}  {level}")

## 5 · Standard clause library (RAG)

We store **market-standard versions** of each clause type.  When analyzing a
contract, we retrieve the most similar standard clause and measure **deviation**.

This is a classic RAG pattern:
1. **Index** standard clauses with embeddings
2. **Retrieve** nearest standard for each analyzed clause
3. **Compare** to quantify how far the clause deviates from the norm

In [ ]:
# ── Standard clause library ──
STANDARD_CLAUSES: Dict[str, str] = {
    "indemnification": (
        "Each party shall indemnify, defend, and hold harmless the other party from "
        "third-party claims arising from the indemnifying party's material breach or "
        "negligence, subject to the limitation of liability set forth herein."
    ),
    "limitation_of_liability": (
        "Neither party's aggregate liability shall exceed the total fees paid or payable "
        "under this Agreement in the twelve months preceding the claim. Neither party shall "
        "be liable for indirect, incidental, consequential, or punitive damages."
    ),
    "ip_assignment": (
        "Vendor retains all rights in pre-existing intellectual property. Work product "
        "created specifically for Client under this Agreement shall be assigned to Client "
        "upon receipt of full payment for the applicable deliverable."
    ),
    "termination": (
        "Either party may terminate this Agreement for convenience with 60 days prior "
        "written notice, or immediately upon material breach that remains uncured for "
        "30 days after written notice."
    ),
    "confidentiality": (
        "Each party shall maintain the confidentiality of the other party's proprietary "
        "information for five years following disclosure. Standard carve-outs apply: "
        "publicly available information, independent development, and legal compulsion."
    ),
    "force_majeure": (
        "Neither party shall be liable for delays or failure to perform due to causes "
        "beyond its reasonable control, including natural disasters, war, pandemic, or "
        "government action, provided prompt notice is given."
    ),
}

# Embed standard clauses
standard_types = list(STANDARD_CLAUSES.keys())
standard_texts = list(STANDARD_CLAUSES.values())
standard_embeddings = model.encode(standard_texts, show_progress_bar=False)

def find_standard_and_deviation(clause_text: str) -> Tuple[str, str, float]:
    """Find closest standard clause and compute cosine similarity."""
    emb = model.encode([clause_text], show_progress_bar=False)[0]
    sims = np.dot(standard_embeddings, emb) / (
        np.linalg.norm(standard_embeddings, axis=1) * np.linalg.norm(emb)
    )
    best_idx = int(np.argmax(sims))
    return standard_types[best_idx], standard_texts[best_idx], float(sims[best_idx])

# ── Test: compare risky clauses against standards ──
RISKY_CLAUSES_TEST = [
    "Vendor shall indemnify Client against any and all claims, losses, and "
    "expenses without limitation, arising from any cause whatsoever.",

    "Client's aggregate liability under this Agreement shall not exceed $100. "
    "Vendor's liability is unlimited.",

    "All intellectual property conceived or developed by Vendor during the term, "
    "including pre-existing IP and unrelated inventions, shall belong to Client.",

    "Client may terminate at any time without cause. Vendor may not terminate.",

    "Vendor shall keep all information confidential in perpetuity. No exceptions. "
    "Client has no reciprocal obligation.",
]

print(f"{'Risky clause (truncated)':<55} {'Matched type':<26} {'Similarity':>10} {'Deviation':>10}")
print("─" * 105)
for clause in RISKY_CLAUSES_TEST:
    stype, stext, sim = find_standard_and_deviation(clause)
    deviation = 1.0 - sim
    short = clause[:52] + "…" if len(clause) > 52 else clause
    level = "🟢" if deviation < 0.15 else "🟡" if deviation < 0.25 else "🔴"
    print(f"{short:<55} {stype:<26} {sim:>9.3f}  {level} {deviation:>8.3f}")

print("\n✓ Higher deviation = clause differs more from market standard → higher risk")

## 6 · Report generation design

The final output is a **structured report** with four sections:
1. Executive summary (2–3 sentences)
2. Clause-by-clause table (type, risk score, explanation)
3. Top risks (highest-scoring clauses, ranked)
4. Negotiation points (what to push back on)

Below we define the output schema and a template renderer.

In [ ]:
import json
from typing import Any

# ── Report schema ──
REPORT_SCHEMA: Dict[str, Any] = {
    "executive_summary": {
        "type": "string",
        "description": "2-3 sentence overview of contract risk posture",
    },
    "metadata": {
        "type": "object",
        "properties": {
            "contract_name": {"type": "string"},
            "total_clauses": {"type": "integer"},
            "analysis_date": {"type": "string", "format": "date"},
            "overall_risk_level": {"type": "string", "enum": ["Low", "Medium", "High", "Critical"]},
        },
    },
    "clause_table": {
        "type": "array",
        "items": {
            "clause_index": {"type": "integer"},
            "clause_type": {"type": "string"},
            "risk_score": {"type": "number", "minimum": 1, "maximum": 5},
            "explanation": {"type": "string"},
            "suggested_mitigation": {"type": "string"},
        },
    },
    "top_risks": {
        "type": "array",
        "max_items": 5,
        "items": {
            "rank": {"type": "integer"},
            "clause_text": {"type": "string"},
            "risk_score": {"type": "number"},
            "plain_english_risk": {"type": "string"},
        },
    },
    "negotiation_points": {
        "type": "array",
        "items": {
            "clause_type": {"type": "string"},
            "current_language": {"type": "string"},
            "suggested_alternative": {"type": "string"},
            "talking_points": {"type": "array", "items": {"type": "string"}},
        },
    },
}

def render_report_template(report_data: Dict) -> str:
    """Render a contract analysis report from structured data."""
    lines: List[str] = []
    lines.append("=" * 70)
    lines.append("       CONTRACT RISK ANALYSIS REPORT")
    lines.append("=" * 70)

    meta = report_data.get("metadata", {})
    lines.append(f"Contract:    {meta.get('contract_name', 'N/A')}")
    lines.append(f"Clauses:     {meta.get('total_clauses', 'N/A')}")
    lines.append(f"Date:        {meta.get('analysis_date', 'N/A')}")
    lines.append(f"Risk Level:  {meta.get('overall_risk_level', 'N/A')}")
    lines.append("")

    lines.append("EXECUTIVE SUMMARY")
    lines.append("-" * 70)
    lines.append(report_data.get("executive_summary", "N/A"))
    lines.append("")

    lines.append("TOP RISKS")
    lines.append("-" * 70)
    for risk in report_data.get("top_risks", []):
        lines.append(f"  #{risk['rank']}  [Score: {risk['risk_score']:.1f}]")
        lines.append(f"      {risk['plain_english_risk']}")
    lines.append("")

    lines.append("NEGOTIATION POINTS")
    lines.append("-" * 70)
    for pt in report_data.get("negotiation_points", []):
        lines.append(f"  ▸ {pt['clause_type'].upper()}")
        for tp in pt.get("talking_points", []):
            lines.append(f"    • {tp}")
    lines.append("")
    lines.append("=" * 70)
    return "\n".join(lines)

# ── Demo with mock data ──
mock_report = {
    "metadata": {
        "contract_name": "TechVendor SaaS Agreement",
        "total_clauses": 7,
        "analysis_date": "2025-01-15",
        "overall_risk_level": "High",
    },
    "executive_summary": (
        "This contract contains 7 clauses, of which 3 present elevated risk. "
        "The indemnification clause lacks any liability cap, and the IP assignment "
        "clause captures pre-existing vendor IP. Immediate negotiation is recommended."
    ),
    "top_risks": [
        {"rank": 1, "risk_score": 4.5, "plain_english_risk":
         "Vendor bears unlimited indemnification — no cap on financial exposure"},
        {"rank": 2, "risk_score": 4.2, "plain_english_risk":
         "IP clause captures pre-existing vendor IP, not just project deliverables"},
        {"rank": 3, "risk_score": 3.8, "plain_english_risk":
         "Termination is one-sided — only Client can exit the agreement"},
    ],
    "negotiation_points": [
        {"clause_type": "indemnification",
         "talking_points": ["Add a liability cap tied to 12 months of fees",
                            "Limit scope to third-party IP claims only"]},
        {"clause_type": "ip_assignment",
         "talking_points": ["Carve out pre-existing vendor IP",
                            "Limit assignment to project-specific deliverables"]},
    ],
}

print(render_report_template(mock_report))
print("\n✓ Report schema defined — ready for LLM-generated content in notebook 02")

## Exercises

1. **Add a clause type** — Add `non_compete` to the exemplar set. Write 2–3
   exemplar sentences, re-run the classifier, and test with 3 non-compete clauses.
   Does accuracy improve?

2. **Tune risk weights** — Adjust `WEIGHTS` to give more importance to
   `party_asymmetry`.  How do the scores change for the 10 example clauses?
   Which weighting feels most aligned with legal intuition?

3. **Expand the standard library** — Add `payment_terms`, `warranty`, and
   `dispute_resolution` standards.  Test deviation scoring on 5 new clauses.
   Do the similarity scores align with your expectations?

## Key Takeaways

| # | Takeaway |
|---|---|
| 1 | A clear data model (Clause → ClassifiedClause → ScoredClause → Deviation → NegotiationPoint) enables modular design |
| 2 | Structural cues (section numbers, paragraph breaks) suffice for clause segmentation |
| 3 | Few-shot embedding classification works well without fine-tuning |
| 4 | Multi-factor risk scoring is more robust than single-signal approaches |
| 5 | RAG-based standard comparison grounds risk assessment in market norms |
| 6 | A structured report schema makes output consistent and parseable |

## What's Next

In **02 — Build** we implement the full end-to-end pipeline with OpenAI
structured outputs, synthetic contracts, and a complete risk report.

## References

1. Harvey AI — *Building AI for Lawyers*, https://www.harvey.ai/blog
2. Zhong, H. et al. — *How Does NLP Benefit Legal System?*, ACL 2020
3. ChromaDB documentation — https://docs.trychroma.com
4. OpenAI Structured Outputs — https://platform.openai.com/docs/guides/structured-outputs